# Charity Dataset Generator — Groq (Free Tier)
Uses **Groq + Llama 4 Scout** for vision. Free tier, no billing required.

**Free tier limits for llama-4-scout:** 30 RPM · 1,000 RPD · 6,000 TPM  
**Your job:** 58 images x 5 questions = 290 calls — fits in one day.

**Get a free API key:** https://console.groq.com/keys

**Steps:** Install → API key → Upload files → Extract → Generate → Save → Download

## 1. Install Dependencies

In [ ]:
!pip install -q groq pillow tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.9 MB/s eta 0:00:00


## 2. Imports

In [ ]:
import os, json, zipfile, time, re, io, base64
from pathlib import Path
from PIL import Image
from tqdm.notebook import tqdm
from groq import Groq
print('Imports OK')

Imports OK


## 3. Configure API Key

In [ ]:
from google.colab import userdata

# Store key in Colab Secrets (lock icon in sidebar) as GROQ_API_KEY
# Get a free key at: https://console.groq.com/keys
try:
    GROQ_API_KEY = userdata.get('KEY')
    print('Loaded from Colab Secrets.')
except Exception:
    GROQ_API_KEY = 'KEY'  # fallback
    print('Using hardcoded key.')

client = Groq(api_key=GROQ_API_KEY)
MODEL  = 'meta-llama/llama-4-scout-17b-16e-instruct'
print('Groq client ready | model:', MODEL)

Using hardcoded key.
Groq client ready | model: meta-llama/llama-4-scout-17b-16e-instruct


## 4. Upload Files

In [ ]:
from google.colab import files
print('Upload images.zip and question_templates.json')
uploaded = files.upload()
for f in uploaded:
    print('  Uploaded:', f)

Upload images.zip and question_templates.json


Saving ds.zip to ds.zip
Saving vqa_questions.json to vqa_questions.json
  Uploaded: ds.zip
  Uploaded: vqa_questions.json


## 5. Extract Images

In [ ]:
ZIP_FILE    = 'ds.zip'
EXTRACT_DIR = 'dataset_images'

os.makedirs(EXTRACT_DIR, exist_ok=True)
with zipfile.ZipFile(ZIP_FILE, 'r') as z:
    z.extractall(EXTRACT_DIR)

image_files = sorted(
    f for f in os.listdir(EXTRACT_DIR)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
)
print('Found', len(image_files), 'images.')

Found 58 images.


## 6. Load Question Templates

In [ ]:
with open('vqa_questions.json', 'r', encoding='utf-8') as f:
    QUESTION_TEMPLATES = json.load(f)

print('Loaded', len(QUESTION_TEMPLATES), 'categories:')
for cat, qs in QUESTION_TEMPLATES.items():
    print(' ', cat, ':', len(qs), 'questions')

Loaded 5 categories:
  Basic Needs & Living Essentials : 5 questions
  Medical & Health Support : 5 questions
  Education & Specialized Support : 5 questions
  Emergency & Disaster Assistance : 5 questions
  Livelihood & Economic Empowerment : 5 questions


## 7. Helper Functions

In [ ]:
# Free tier: 30 RPM = 1 request per 2s minimum. Use 3s to be safe.
MIN_DELAY   = 3.0
MAX_RETRIES = 4

_last_call  = 0.0


def get_category(filename):
    stem = Path(filename).stem.lower()
    for cat in QUESTION_TEMPLATES:
        if cat.lower() in stem:
            return cat
    return list(QUESTION_TEMPLATES.keys())[0]


def image_to_base64(image_path, max_px=1024):
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    if max(w, h) > max_px:
        scale = max_px / max(w, h)
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=85)
    return base64.b64encode(buf.getvalue()).decode('utf-8')


def parse_retry_after(exc):
    try:
        m = re.search(r'retry.after[^\d]+(\d+\.?\d*)', str(exc), re.IGNORECASE)
        if m:
            return min(float(m.group(1)) + 1, 90.0)
    except Exception:
        pass
    return 30.0


def ask_groq(b64_image, question):
    global _last_call

    prompt = (
        'قم بتحليل صورة أو مستند طلب المساعدة الخيرية.\n\n'
        'أجب فقط بناءً على الصورة.\n\n'
        'السؤال:\n' + question + '\n\n'
        'أعط إجابة واقعية مختصرة باللغة العربية.'
    )

    for attempt in range(1, MAX_RETRIES + 1):
        # throttle
        gap = time.time() - _last_call
        if gap < MIN_DELAY:
            time.sleep(MIN_DELAY - gap)

        try:
            _last_call = time.time()
            response = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        'role': 'user',
                        'content': [
                            {
                                'type': 'image_url',
                                'image_url': {
                                    'url': 'data:image/jpeg;base64,' + b64_image
                                }
                            },
                            {'type': 'text', 'text': prompt}
                        ]
                    }
                ],
                max_tokens=512,
            )
            return response.choices[0].message.content.strip()

        except Exception as e:
            err = str(e)
            is_rate = '429' in err or 'rate_limit' in err.lower() or 'rate limit' in err.lower()
            if is_rate and attempt < MAX_RETRIES:
                wait = parse_retry_after(e)
                print('  429 - waiting {}s (attempt {}/{})...'.format(int(wait), attempt + 1, MAX_RETRIES))
                time.sleep(wait)
            else:
                raise


print('Helpers ready | throttle:', MIN_DELAY, 's/call')

Helpers ready | throttle: 3.0 s/call


## 8. Resume — Load Existing Progress

In [ ]:
OUTPUT_FILE = 'dataset.json'

if os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        dataset = json.load(f)
    done_ids = set(r['image_id'] for r in dataset)
    print('Resuming:', len(dataset), 'records,', len(done_ids), 'images already done.')
else:
    dataset  = []
    done_ids = set()
    print('Starting fresh.')

Starting fresh.


## 9. Generate Dataset

In [ ]:
errors     = []
SAVE_EVERY = 5

total_q = sum(
    len(QUESTION_TEMPLATES.get(get_category(f), list(QUESTION_TEMPLATES.values())[0]))
    for f in image_files
)
remaining = sum(
    len(QUESTION_TEMPLATES.get(get_category(f), list(QUESTION_TEMPLATES.values())[0]))
    for i, f in enumerate(image_files, 1) if i not in done_ids
)
est_min = remaining * MIN_DELAY / 60
print('{} images | {} total calls | {} remaining | ~{:.0f} min'.format(
    len(image_files), total_q, remaining, est_min))
print('Skipping', len(done_ids), 'already-done images.\n')

start = time.time()

for image_id, file in enumerate(tqdm(image_files, desc='Images'), start=1):

    if image_id in done_ids:
        continue

    image_path = os.path.join(EXTRACT_DIR, file)
    category   = get_category(file)
    questions  = QUESTION_TEMPLATES.get(category, list(QUESTION_TEMPLATES.values())[0])

    print('[{:02d}/{}] {}  ->  {}'.format(image_id, len(image_files), file, category))

    try:
        b64 = image_to_base64(image_path)
    except Exception as e:
        print('  Could not load image:', e)
        errors.append('img {}: load error: {}'.format(image_id, e))
        continue

    for q_idx, question in enumerate(questions, 1):
        try:
            answer = ask_groq(b64, question)
            dataset.append({
                'image_id':   image_id,
                'type':       category,
                'question':   question,
                'answer':     answer,
                'image_path': '/images/' + file,
            })
            print('  Q{}:'.format(q_idx), answer[:60], '...' if len(answer) > 60 else '')
        except Exception as e:
            msg = '[img {}] Q{}: {}'.format(image_id, q_idx, str(e)[:100])
            print('  WARNING:', msg)
            errors.append(msg)

    if image_id % SAVE_EVERY == 0:
        with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
            json.dump(dataset, f, ensure_ascii=False, indent=2)
        print('  [auto-saved {} records]'.format(len(dataset)))

elapsed = time.time() - start
print('\nDone in {:.1f} min | {} records | {} errors'.format(
    elapsed / 60, len(dataset), len(errors)))

58 images | 290 total calls | 290 remaining | ~14 min
Skipping 0 already-done images.



Images:   0%|          | 0/58 [00:00<?, ?it/s]

[01/58] Basic Needs & Living Essentials (1).jpeg  ->  Basic Needs & Living Essentials
  Q1: تحليل الصورة:

المستفيدون يحتاجون إلى المواد المعروضة في الص ...
  Q2: من خلال الصورة المقدمة، يبدو أن هناك حاجة إلى بعض المواد الأ ...
  Q3: المستفيدون من المساعدة الخيرية في هذه الصورة هم على الأرجح أ ...
  Q4: تحليل صورة أو مستند طلب المساعدة الخيرية:

الصورة المرفقة لط ...
  Q5: نعم، هناك عدة متطلبات واعتبارات للوصول إلى ظروف معيشة لائقة  ...
[02/58] Basic Needs & Living Essentials (1).jpg  ->  Basic Needs & Living Essentials
  Q1: تحليل الصورة:

الصورة تظهر مبنى متضررًا بشكل كبير، يبدو أنه  ...
  Q2: تحليل الصورة:

المواد المطلوبة:

1.  **أسمنت**: كمية غير محد ...
  Q3: تحليل الصورة:

الصورة تُظهر مبنى متضررًا بشكل كبير، مما يشير ...
  Q4: الجواب هو: 
لا يوجد مستندات مرفقة. 
تدعم الصورة الطلب كونها  ...
  Q5: نعم، هناك متطلبات خاصة: يجب أن تكون المساعدات مقدمة بشكل سري ...
[03/58] Basic Needs & Living Essentials (1).png  ->  Basic Needs & Living Essentials
  Q1: تحليل الصورة:

المستفيدون في

KeyboardInterrupt: 

## 10. Save Final Dataset

In [ ]:
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, ensure_ascii=False, indent=2)

print('Saved', len(dataset), 'records ->', OUTPUT_FILE)
if errors:
    print('\nFailed calls:')
    for e in errors:
        print(' ', e)

Saved 149 records -> dataset.json

Failed calls:
  [img 27] Q5: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-
  [img 28] Q1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-
  [img 28] Q2: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-
  [img 28] Q4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-
  [img 28] Q5: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-
  [img 29] Q1: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-
  [img 29] Q2: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-
  [img 29] Q4: Error code: 429 - {'error': {'message': 'Rate limit reached for model `meta-llama/llama-4-scout-17b-
  [img 29] Q5: Error co

## 11. Preview & Download

In [ ]:
print(json.dumps(dataset[:3], ensure_ascii=False, indent=2))

[
  {
    "image_id": 1,
    "type": "Basic Needs & Living Essentials",
    "question": "لماذا يحتاج المستفيدون لهذه المواد؟ (سبب الطلب)",
    "answer": "تحليل الصورة:\n\nالمستفيدون يحتاجون إلى المواد المعروضة في الصورة لأنهم ربما يعانون من صعوبات اقتصادية أو اجتماعية تجعل من الصعب عليهم الحصول على هذه المواد الأساسية بأنفسهم. يمكن أن يكونوا من الأسر ذات الدخل المنخفض أو اللاجئين أو الأشخاص الذين يعيشون في مناطق تعاني من تدهور اقتصادي أو سياسي.\n\nالطلب على المواد المعروضة في الصورة قد يكون ناتجًا عن عدة أسباب، مثل:\n\n1.  **عدم وجود مصدر دخل ثابت:** قد يكون الأشخاص غير قادرين على شراء هذه السلع بأنفسهم بسبب عدم وجود مصدر دخل ثابت أو بسبب البطالة.\n2.  **تدهور الأوضاع الاقتصادية:** في بعض الأحيان، يمكن أن تؤدي الأزمات الاقتصادية أو السياسية إلى نقص في المواد الغذائية الأساسية وزيادة في الأسعار، مما يجعل من الصعب على بعض الأشخاص الحصول عليها.\n3.  **إعصار أو كارثة طبيعية:** يمكن أن تؤدي الكوارث الطبيعية إلى تدمير البنية التحتية وتهجير الناس، مما يتركهم بدون إمكانية الوصول إلى المواد الأ

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>